# Cohort comparison: 6 cohorts x 2 endpoints

Read-only. Refits nothing — run `01`/`02`/`03` first for the same cross this
notebook reads.

The modeled cross is **cohort x exclusion x endpoint**, which is 3 cohorts x 2
exclusions = **6 patient cohorts**, each fit for **2 endpoints**:

| axis | values |
|---|---|
| cohort | `all`, `metastatic_adt` (ADT-intent), `metastatic_llm` (met_diagnosis LLM) |
| exclusion | `none`, `pre_adt_castrate` (`_noprecastrate`) |
| endpoint | `platinum` (`PLATINUM` / `t_platinum`), `nepc` (`NEPC` / `t_nepc`) |

All 12 cells share the ADT anchor (first ADT exposure = time 0). This notebook
reports the **+180d landmark only** — the most consistent signal across the
cross. Widen `LANDMARKS` in the config cell to sweep `0 / 90 / 180` again.

### What this notebook reports

1. **Cohort numbers + event incidence** — per cohort x endpoint, at +180d.
2. **PSA and testosterone** — pre- and post-treatment levels, stratified by event.
3. **Univariate associations** for PSA/testosterone across every cohort x endpoint.
4. **Effect of narrowing the cohort** — `all` -> metastatic -> `+noprecastrate`.

### Three things to keep in view while reading

**These are not the same patients across endpoints.** Each endpoint's build
applies only its own time-validity gate (`t_platinum > 0` never filters NEPC and
`t_nepc > 0` never filters platinum), so a platinum-vs-NEPC metric difference
confounds *event* with *cohort*. Section 1's overlap table quantifies it.

**The cohort axis is nested, the exclusion axis is orthogonal.** `metastatic_*`
is a subset of `all`; `_noprecastrate` composes onto any cohort by removing
patients with a castrate testosterone (<50 ng/dL) strictly before ADT start.
Section 4 walks the cascade so shrinking N is never mistaken for a real effect.

**The NEPC label here is the criteria-timeline component**, narrower than the
"any NE feature -> NEPC" rule used by the binary classifier in the Figure 2
enrichment analysis. The two are **not** interchangeable; name which one you
mean in any text drawn from this notebook.

In [ ]:
from pathlib import Path
import itertools
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, ".")
import compass_pipeline as cp
import cox_aggregated as ca

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------------
# The cross. Keep these in lockstep with 01/02/03 -- this notebook reads the
# trees those notebooks write, and make_endpoint_runs() is the same factory,
# so the directory names cannot drift between them.
# ---------------------------------------------------------------------------
ARMS = ["adt"]
ENDPOINTS = ("platinum", "nepc")
COHORTS = ("all", "metastatic_adt", "metastatic_llm")
EXCLUSIONS = ("none", "pre_adt_castrate")
# Single landmark for the whole notebook: +180d carries the most consistent
# signal across the cross, so every section below reports it and only it. Set
# this to a tuple like (0, 90, 180) to widen the sweep again -- every table is
# driven off LANDMARKS, and the per-section knobs derive from LANDMARK.
LANDMARK = 180
LANDMARKS = (LANDMARK,)
ID_COL = "DFCI_MRN"

# Override if the COMPASS output tree is mounted elsewhere.
DATA_ROOT = cp._PROFILE_OUTPUT_ROOT
SURVIVAL_ROOT = DATA_ROOT / "survival_analysis"

RUNS = cp.make_endpoint_runs(
    ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=EXCLUSIONS
)
print(f"\n{len(RUNS)} runs = {len(COHORTS)} cohorts x {len(EXCLUSIONS)} exclusions "
      f"x {len(ENDPOINTS)} endpoints x {len(ARMS)} arm(s)")

# ---------------------------------------------------------------------------
# Figure output and shared plot styling. Defined here, not in section 3, because
# sections 1 and 3 both draw and a fresh top-to-bottom run must not depend on
# which one happens to execute first.
#
# Same root and <ARM>/ layout as 05_figures.Rmd and the R pipeline, so this
# notebook's supplements land beside the generated figure set rather than in a
# parallel tree. Override with COMPASS_FIG_ROOT, matching
# export_adt_intent_outputs.py.
# ---------------------------------------------------------------------------
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

FIG_ROOT = Path(os.environ.get(
    "COMPASS_FIG_ROOT", "/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS"
))
SUPPLEMENT_DIR = FIG_ROOT / ARMS[0].upper() / "supplements"
SAVE_DPI = 600

# Endpoint identity colors: categorical slots 1 and 2, validated for CVD
# separation (worst protan/deutan OKLab dE 24.7, well above the target of 8).
ENDPOINT_COLOR = {"platinum": "#2a78d6", "nepc": "#eb6834"}
FALLBACK_COLOR = "#52514e"


def save_supplement(fig, stem):
    """Write one supplement figure, creating the directory on first use."""
    try:
        SUPPLEMENT_DIR.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        # A local run without the share mounted should still show the figure.
        print(f"!! could not create {SUPPLEMENT_DIR} ({exc}); figure not saved.")
        return None
    out = SUPPLEMENT_DIR / f"{stem}.png"
    fig.savefig(out, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white")
    print(f"[fig ] {out}")
    return out


print(f"supplement figures -> {SUPPLEMENT_DIR}")


In [ ]:
# Human-readable identity for one run. `cohort_key` is the PATIENT cohort
# (cohort x exclusion) -- the 6 -- and is endpoint-independent by construction,
# so the same key lines up across endpoints in every table below.
def cohort_key(run):
    return f"{run['cohort']}|{run['exclusion']}"


def cohort_label(run):
    excl = "" if run["exclusion"] == "none" else " +noprecastrate"
    return f"{run['cohort']}{excl}"


# Cohort ordering used by every table: the nesting cascade of section 4.
COHORT_ORDER = [
    f"{c}{'' if e == 'none' else ' +noprecastrate'}"
    for c in COHORTS for e in EXCLUSIONS
]
ENDPOINT_ORDER = list(ENDPOINTS)


def order_cohorts(frame, col="cohort"):
    """Sort a table into COHORT_ORDER without dropping unexpected labels."""
    if col not in frame.columns:
        return frame
    out = frame.copy()
    out[col] = pd.Categorical(out[col], categories=COHORT_ORDER, ordered=True)
    sort_cols = [col] + [c for c in ("endpoint", "landmark_days") if c in out.columns]
    return out.sort_values(sort_cols).reset_index(drop=True)


def normalize_id(series):
    """String-normalize MRNs so int/float round-trips don't break merges."""
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype("string")


# Collect missing artifacts and report them at the end rather than crashing: a
# partially complete run should still show whatever it does have.
MISSING = []


def require(path, what):
    if Path(path).exists():
        return True
    MISSING.append(f"{what}: {path}")
    return False


def agg_path(run, landmark):
    return run["inputs_dir"] / f"aggregated_landmark{landmark}.csv"


# Endpoint -> (event_col, duration_col), straight from the shared registry so
# this notebook cannot invent a naming convention of its own.
EVENT_COL = {ep: ca.ENDPOINTS[ep]["event_col"] for ep in ENDPOINTS}
DURATION_COL = {ep: ca.ENDPOINTS[ep]["duration_col"] for ep in ENDPOINTS}
print("endpoint -> (event, duration):",
      {ep: (EVENT_COL[ep], DURATION_COL[ep]) for ep in ENDPOINTS})

## 1. Cohort numbers and event incidence

**The most important section in this notebook.** Every model comparison below is
conditioned on these counts: if an endpoint has a double-digit event count in a
narrowed cohort, differences downstream are statements about power, not biology.

`1a` is the Stage 1 cohort before landmark gating; `1b` is what each model
actually fit; `1c` is the endpoint overlap within each patient cohort.

In [ ]:
# --- 1a. Stage 1: the shared ADT cohort, before cohort/exclusion/landmark gating
cohort_stage1 = None
# cohort="all" resolves to the arm's own Stage-1 survival cohort CSV.
stage1_path = cp.cohort_mrn_list_path("all", arm="adt", data_root=DATA_ROOT)

if require(stage1_path, "Stage 1 survival cohort"):
    cohort_stage1 = pd.read_csv(stage1_path, low_memory=False)
    cohort_stage1[ID_COL] = normalize_id(cohort_stage1[ID_COL])

    rows = []
    for ep in ENDPOINTS:
        event_col = EVENT_COL[ep]
        if event_col not in cohort_stage1.columns:
            rows.append({"endpoint": ep, "n_patients": len(cohort_stage1),
                         "note": f"{event_col} absent from the Stage 1 cohort"})
            continue
        events = pd.to_numeric(cohort_stage1[event_col], errors="coerce").fillna(0)
        # Prevalent = event at or before the anchor; the incident gate drops these,
        # which is the main reason a landmark cohort is smaller than Stage 1.
        # Stage 1 and Stage 3 use DIFFERENT duration-column conventions, and this
        # cell reads Stage 1. The cohort CSV carries TT_PLATINUM / TT_NEPC (see
        # compile_COMPASS_cohort_data.py); the lowercase t_platinum / t_nepc in
        # ca.ENDPOINTS["duration_col"] are the Stage 3 names, valid only in the
        # aggregated_landmark*.csv files section 1b reads. Using DURATION_COL[ep]
        # here silently finds nothing.
        duration_col = f"TT_{EVENT_COL[ep]}"
        # Guard on the COLUMN, not the converted result: .get() on a missing
        # column returns None, and pd.to_numeric(None) is a scalar NaN rather
        # than a Series, which has no .notna()/.le().
        if duration_col in cohort_stage1.columns:
            tt = pd.to_numeric(cohort_stage1[duration_col], errors="coerce")
            n_prevalent = int((events.eq(1) & tt.le(0)).sum())
        else:
            n_prevalent = np.nan
        rows.append({
            "endpoint": ep,
            "n_patients": len(cohort_stage1),
            "n_events": int(events.eq(1).sum()),
            "event_rate_pct": 100 * events.eq(1).mean(),
            "n_prevalent_at_anchor": n_prevalent,
            "note": "" if duration_col in cohort_stage1.columns
                    else f"{duration_col} absent; prevalent count unavailable",
        })
    print(f"Stage 1 ADT cohort: {len(cohort_stage1):,} patients "
          f"(shared by all 6 cohorts, before any restriction)\n")
    display(pd.DataFrame(rows).round(2))
else:
    print("Stage 1 cohort not found; 1a skipped.")

In [ ]:
# --- 1b. Per-cohort x endpoint x landmark: what each model actually fit
landmark_rows = []
for run in RUNS:
    ep = run["endpoint"]
    event_col, duration_col = EVENT_COL[ep], DURATION_COL[ep]
    for lm in LANDMARKS:
        base = {"cohort": cohort_label(run), "endpoint": ep, "landmark_days": lm}
        path = agg_path(run, lm)
        if not require(path, f"{run['label']} [{ep}] aggregated landmark {lm}"):
            landmark_rows.append({**base, "status": "missing"})
            continue
        agg = pd.read_csv(path, low_memory=False)
        if event_col not in agg.columns:
            landmark_rows.append({**base, "status": f"no {event_col} column"})
            continue
        events = pd.to_numeric(agg[event_col], errors="coerce").fillna(0)
        # Same guard as 1a: only convert when the column is actually present.
        has_duration = duration_col in agg.columns
        durations = (pd.to_numeric(agg[duration_col], errors="coerce")
                     if has_duration else None)
        landmark_rows.append({
            **base,
            "n_patients": len(agg),
            "n_events": int(events.eq(1).sum()),
            "event_rate_pct": 100 * events.eq(1).mean(),
            "median_days_to_event": durations.loc[events.eq(1)].median()
                                    if has_duration else np.nan,
            "median_days_followup": durations.median() if has_duration else np.nan,
            "status": "ok" if has_duration else f"ok ({duration_col} absent)",
        })

counts = order_cohorts(pd.DataFrame(landmark_rows))
display(counts.round(1))

In [ ]:
# Event counts as a compact cohort x endpoint grid, one block per landmark.
ok_counts = counts.loc[counts["status"].eq("ok")] if "status" in counts.columns else counts
if not ok_counts.empty:
    for value, title in (("n_events", "events"), ("n_patients", "patients"),
                         ("event_rate_pct", "event rate (%)")):
        grid = ok_counts.pivot_table(
            index="cohort", columns=["endpoint", "landmark_days"],
            values=value, observed=False,
        ).reindex(COHORT_ORDER).dropna(how="all")
        print(f"\n=== {title} ===")
        display(grid.round(1))

    # Rules of thumb put stable multivariable Cox at ~10 events per covariate, so
    # flag any cell that cannot support the lab feature set no matter its C-index.
    THIN_EVENTS = 25
    thin = ok_counts.loc[ok_counts["n_events"].lt(THIN_EVENTS)]
    if not thin.empty:
        print(f"\n!! {len(thin)} cell(s) with < {THIN_EVENTS} events -- underpowered, "
              "read metrics there as power, not biology:")
        display(thin[["cohort", "endpoint", "landmark_days", "n_patients", "n_events"]])
    else:
        print(f"\nAll cells have >= {THIN_EVENTS} events.")

### 1c. Endpoint overlap within each patient cohort

How much of a platinum-vs-NEPC difference is *cohort* rather than *event*. Both
endpoints start from the same patient cohort but each applies its own incident
gate, so the surviving sets differ.

In [ ]:
# Index runs by (cohort_key, endpoint) so the two endpoints of one patient
# cohort can be compared directly.
RUN_BY = {(cohort_key(r), r["endpoint"]): r for r in RUNS}

overlap_rows = []
for ckey in dict.fromkeys(cohort_key(r) for r in RUNS):
    for lm in LANDMARKS:
        ids, label = {}, None
        for ep in ENDPOINTS:
            run = RUN_BY.get((ckey, ep))
            if run is None:
                continue
            label = cohort_label(run)
            path = agg_path(run, lm)
            if not path.exists():
                continue
            frame = pd.read_csv(path, usecols=[ID_COL], low_memory=False)
            ids[ep] = set(normalize_id(frame[ID_COL]).dropna())
        if len(ids) < 2:
            continue
        a, b = ENDPOINTS[0], ENDPOINTS[1]
        sa, sb = ids[a], ids[b]
        overlap_rows.append({
            "cohort": label, "landmark_days": lm,
            f"n_{a}": len(sa), f"n_{b}": len(sb),
            "n_shared": len(sa & sb),
            f"n_{a}_only": len(sa - sb), f"n_{b}_only": len(sb - sa),
            f"pct_of_{a}_retained": 100 * len(sa & sb) / len(sa) if sa else np.nan,
        })

if overlap_rows:
    display(order_cohorts(pd.DataFrame(overlap_rows)).round(1))
else:
    print("Both endpoint trees are needed for the overlap table; see missing artifacts.")

### 1d. ADT-intent vs LLM metastatic labels

`metastatic_adt` and `metastatic_llm` are two independent answers to "is this
patient metastatic", and the 6-cohort cross treats them as interchangeable
axes. This section asks how much they actually agree.

The two label sets do **not** cover the same denominator, and that asymmetry
drives everything below:

| | source | coverage |
|---|---|---|
| `ADT_INTENT` | medication-derived classifier (`classify_adt_intent`) | **every** Stage-1 patient -- `build_adt_intent_mrn_lists()` raises if any is unlabelled |
| `LLM_METASTATIC` | `met_diagnosis` LLM adjudication | only patients the LLM run covered -- `build_llm_met_mrn_lists()` drops unlabelled patients *before writing the file*, so they are **absent**, not null |

So an LLM-unlabelled patient is not an LLM-negative patient. A plain 2x2 would
silently merge those two groups and overstate agreement, which is why the cell
below left-joins the LLM table onto the (complete) ADT table and carries an
explicit `(no LLM label)` column. Read the agreement
statistics as applying only to the jointly-labelled subset, and read the
coverage line as telling you how much of the cohort that subset actually is.

In [ ]:
# --- 1d. Do the two metastatic definitions agree?
# Reads the LABEL tables (not the derived MRN lists) so unlabelled patients stay
# visible: the MRN lists only contain positives, which cannot distinguish
# "LLM says no" from "LLM never saw this patient".
merged = None
mrn_dir = DATA_ROOT / "mrn_lists"
adt_labels_path = mrn_dir / cp.ADT_INTENT_LABELS_FILENAME
llm_labels_path = mrn_dir / cp.LLM_MET_LABELS_FILENAME

if require(adt_labels_path, "ADT-intent labels") and require(llm_labels_path, "LLM metastatic labels"):
    adt_lab = pd.read_csv(adt_labels_path)
    llm_lab = pd.read_csv(llm_labels_path)
    adt_lab["DFCI_MRN"] = normalize_id(adt_lab["DFCI_MRN"])
    llm_lab["DFCI_MRN"] = normalize_id(llm_lab["DFCI_MRN"])

    # Left join on the ADT-intent side: it is the complete Stage-1 cohort, so
    # every unmatched row is a genuine LLM coverage gap rather than a bad merge.
    keep_llm = ["DFCI_MRN", "LLM_METASTATIC"] + [
        c for c in ("label_source",) if c in llm_lab.columns
    ]
    merged = adt_lab.merge(llm_lab[keep_llm], on="DFCI_MRN", how="left")

    unexpected = set(merged["ADT_INTENT"].dropna().unique()) - {
        "METASTATIC", "LOCALIZED_ADJUVANT"
    }
    if unexpected:
        print(f"WARNING: unexpected ADT_INTENT values, counted as non-metastatic: {sorted(unexpected)}")
    merged["adt_met"] = merged["ADT_INTENT"].eq("METASTATIC")
    # Tri-state on purpose: True / False / not-in-the-file are three different
    # things. build_llm_met_mrn_lists() DROPS unlabelled patients before writing,
    # so the missingness comes from the left join above, not from nulls in the
    # column -- which is exactly why we joined onto the ADT side.
    # polars writes booleans as true/false, which pandas reads as a real bool
    # column here but as object if anything else ever lands in it; handle both.
    llm_raw = merged["LLM_METASTATIC"]
    if llm_raw.dtype == object or pd.api.types.is_string_dtype(llm_raw):
        llm_raw = llm_raw.astype("string").str.strip().str.lower().map(
            {"true": True, "false": False, "1": True, "0": False,
             "yes": True, "no": False}
        )
    merged["llm_met"] = llm_raw.astype("boolean")

    n_total = len(merged)
    n_labelled = int(merged["llm_met"].notna().sum())
    print(f"Stage 1 ADT cohort:        {n_total:,} patients")
    print(f"  ADT-intent labelled:     {int(merged['ADT_INTENT'].notna().sum()):,}"
          f"  ({merged['ADT_INTENT'].notna().mean():.1%})")
    print(f"  LLM labelled:            {n_labelled:,}  ({n_labelled / n_total:.1%})"
          f"   <- agreement stats below are on THIS subset only")
    if "label_source" in merged.columns:
        src = merged.loc[merged["llm_met"].notna(), "label_source"].value_counts(dropna=False)
        for name, count in src.items():
            print(f"    label_source={name}: {count:,}")

    # --- crosstab, unlabelled kept as its own column ---
    adt_axis = merged["adt_met"].map({True: "ADT: metastatic", False: "ADT: localized"})
    llm_axis = merged["llm_met"].map({True: "LLM: metastatic", False: "LLM: non-metastatic"})
    llm_axis = llm_axis.astype("object").where(merged["llm_met"].notna(), "(no LLM label)")

    ct = pd.crosstab(
        adt_axis.rename("ADT intent"), llm_axis.rename("LLM met_diagnosis")
    )
    ct = ct.reindex(
        index=["ADT: metastatic", "ADT: localized"],
        columns=["LLM: metastatic", "LLM: non-metastatic", "(no LLM label)"],
    ).fillna(0).astype(int)
    ct["total"] = ct.sum(axis=1)
    ct.loc["total"] = ct.sum(axis=0)
    print("\nLabel crosstab (whole Stage 1 cohort):")
    display(ct)

    # --- agreement, jointly-labelled subset only ---
    both = merged.loc[merged["llm_met"].notna()].copy()
    if both.empty:
        print("\nNo jointly-labelled patients -- agreement statistics skipped.")
    else:
        a = both["adt_met"].to_numpy(dtype=bool)
        l = both["llm_met"].to_numpy(dtype=bool)
        n11, n10, n01, n00 = (
            int((a & l).sum()), int((a & ~l).sum()),
            int((~a & l).sum()), int((~a & ~l).sum()),
        )
        n = n11 + n10 + n01 + n00
        po = (n11 + n00) / n
        # Cohen's kappa: raw agreement is inflated when one label is rare, which
        # it is here (most of an ADT cohort is metastatic by intent).
        pe = ((n11 + n10) * (n11 + n01) + (n01 + n00) * (n10 + n00)) / (n * n)
        kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
        jaccard = n11 / (n11 + n10 + n01) if (n11 + n10 + n01) else float("nan")

        # Directional: neither label is ground truth, so report BOTH conditionals
        # rather than picking one as the reference standard.
        print(f"\nAgreement on the {n:,} jointly-labelled patients:")
        print(f"  raw agreement        {po:.1%}   ({n11 + n00:,}/{n:,})")
        print(f"  Cohen's kappa        {kappa:.3f}")
        print(f"  Jaccard (met only)   {jaccard:.1%}   ({n11:,} / {n11 + n10 + n01:,})")
        print(f"  P(LLM met | ADT met)     {n11 / (n11 + n10):.1%}" if (n11 + n10) else "  P(LLM met | ADT met)     n/a")
        print(f"  P(ADT met | LLM met)     {n11 / (n11 + n01):.1%}" if (n11 + n01) else "  P(ADT met | LLM met)     n/a")
        print(f"\n  disagreements: {n10:,} ADT-only metastatic, {n01:,} LLM-only metastatic")

        prev = pd.DataFrame({
            "label": ["ADT intent", "LLM"],
            "n_metastatic": [n11 + n10, n11 + n01],
            "n_labelled": [n, n],
        })
        prev["prevalence"] = prev["n_metastatic"] / prev["n_labelled"]
        print("\nMetastatic prevalence, same denominator:")
        display(prev)


### 1e. Who are the disagreements, and does the choice change the endpoint?

Section 1d says *how much* the two labels agree. This one asks whether the
disagreement is systematic and whether it matters downstream:

1. **Which side is the outlier.** `HAS_POSITIVE_METASTATIC_EVIDENCE` is the
   ADT-intent classifier's own evidence flag, and `label_source` says whether
   the LLM adjudicated a note or fell through to an auto-negative. Cross-tabbing
   those against the disagreement cells says whether the ADT-only calls are
   evidence-backed or intent-inferred, and whether the LLM-only calls come from
   real adjudications or from the auto-negative default.
2. **Whether the endpoints move.** Event rates per agreement cell, off the
   Stage 1 cohort. If the four cells have similar rates, the two definitions are
   interchangeable for this analysis and the 6-cohort cross is carrying a
   redundant axis. If they diverge, the metastatic axis is doing real work and
   the disagreement cells are where the definition choice actually bites.

Event columns come from the **Stage 1** cohort, so the duration columns are
`TT_PLATINUM` / `TT_NEPC`, not the lowercase Stage 3 names -- same convention
note as section 1a. These are crude cumulative rates over the whole observed
record, not landmark-gated incidence; they are here to compare cells against
each other, not to be quoted as incidence.

In [ ]:
# --- 1e. Characterize the disagreements, and check whether they move the endpoint.
# Depends on 1d's `merged` and 1a's `cohort_stage1`.
if merged is None:
    print("Section 1d did not run (missing label tables); skipping.")
elif merged["llm_met"].notna().sum() == 0:
    print("No jointly-labelled patients; skipping.")
else:
    both = merged.loc[merged["llm_met"].notna()].copy()
    both["cell"] = np.select(
        [
            both["adt_met"] & both["llm_met"].astype(bool),
            both["adt_met"] & ~both["llm_met"].astype(bool),
            ~both["adt_met"] & both["llm_met"].astype(bool),
        ],
        ["both metastatic", "ADT only", "LLM only"],
        default="both non-metastatic",
    )
    CELL_ORDER = ["both metastatic", "ADT only", "LLM only", "both non-metastatic"]

    # --- 1. what distinguishes the disagreeing patients ---
    if "HAS_POSITIVE_METASTATIC_EVIDENCE" in both.columns:
        ev = pd.to_numeric(
            both["HAS_POSITIVE_METASTATIC_EVIDENCE"]
            .astype("string").str.strip().str.lower()
            .map({"true": 1, "false": 0, "1": 1, "0": 0}),
            errors="coerce",
        )
        tab = (
            pd.DataFrame({"cell": both["cell"], "has_evidence": ev})
            .groupby("cell", observed=False)["has_evidence"]
            .agg(n="size", n_with_evidence="sum", pct_with_evidence="mean")
        )
        tab["pct_with_evidence"] *= 100
        print("ADT-intent positive metastatic evidence, by agreement cell:")
        print("  (an 'ADT only' row with LOW evidence = intent inferred from the")
        print("   medication pattern alone, which is the weaker of the two claims)")
        display(tab.reindex(CELL_ORDER).dropna(how="all").round(1))

    if "label_source" in both.columns:
        src = pd.crosstab(
            both["cell"], both["label_source"].fillna("(none)"), normalize="index"
        ).mul(100).round(1)
        print("\nLLM label_source composition (% of row), by agreement cell:")
        print("  (an 'LLM only' row dominated by adjudicated notes is a real")
        print("   finding; one dominated by auto_negative would be a default)")
        display(src.reindex([c for c in CELL_ORDER if c in src.index]))

    # --- 2. does the disagreement change the endpoint? ---
    if cohort_stage1 is None:
        print("\nStage 1 cohort unavailable; per-cell event rates skipped.")
    else:
        # Stage 1 duration convention: TT_<ENDPOINT>, not the lowercase Stage 3
        # names in ca.ENDPOINTS["duration_col"]. See the note in section 1a.
        keep = [ID_COL] + [
            c for ep in ENDPOINTS
            for c in (EVENT_COL[ep], f"TT_{ep.upper()}")
            if c in cohort_stage1.columns
        ]
        ev_frame = both[["DFCI_MRN", "cell"]].merge(
            cohort_stage1[keep], left_on="DFCI_MRN", right_on=ID_COL, how="inner"
        )
        n_unmatched = len(both) - len(ev_frame)
        if n_unmatched:
            print(f"\nNOTE: {n_unmatched:,} labelled patients did not match the Stage 1 "
                  "cohort and are excluded from the rates below.")

        rows = []
        for cell, grp in ev_frame.groupby("cell", observed=False):
            row = {"cell": cell, "n_patients": len(grp)}
            for ep in ENDPOINTS:
                col = EVENT_COL[ep]
                if col not in grp.columns:
                    row[f"{ep}_events"] = row[f"{ep}_rate_pct"] = np.nan
                    continue
                e = pd.to_numeric(grp[col], errors="coerce").fillna(0)
                row[f"{ep}_events"] = int((e > 0).sum())
                row[f"{ep}_rate_pct"] = 100 * (e > 0).mean()
            rows.append(row)

        rates = pd.DataFrame(rows).set_index("cell").reindex(CELL_ORDER).dropna(how="all")
        print("\nCrude event rates by agreement cell (whole observed record, NOT landmark-gated):")
        display(rates.round(1))

        # The headline: if the two 'only' cells look like their respective
        # agreeing cells, the label choice is cosmetic for this endpoint.
        for ep in ENDPOINTS:
            c = f"{ep}_rate_pct"
            if c not in rates.columns or rates[c].isna().all():
                continue
            spread = rates[c].max() - rates[c].min()
            print(f"  {ep}: rate spread across cells = {spread:.1f} pp "
                  f"({rates[c].idxmin()} {rates[c].min():.1f}% -> "
                  f"{rates[c].idxmax()} {rates[c].max():.1f}%)")


### 1f. The label overlap, as a figure

The same numbers as 1d and 1e, in the form people actually read them. Two panels:

**(a) Label overlap.** A stacked bar per ADT-intent class, split by the LLM call.
Deliberately *not* a Venn diagram: the `(no LLM label)` group is a coverage gap,
not a set-intersection region, and a Venn would either hide it or misdraw it as
an overlap. The gray segment is that gap, and its width is the honest answer to
"how much of the cohort can these two labels even be compared on."

**(b) Patients per agreement cell.** The four cells, sized. The two disagreement
cells are the ones to watch — if they are thin, the disagreement is a rounding
error on the cohort rather than a real fork in the definition.

Event incidence is deliberately **not** here. It is a property of the analysis
cohorts, not of this label comparison, so it gets its own figure in 1g.

In panel (a) the LLM call is an ordered three-step ramp (metastatic →
non-metastatic → no label), not categorical hues, because the third category is
an absence rather than a peer.

In [ ]:
# --- 1f. Plot the label overlap, cohort sizes, and event incidence.
# Reuses 1d's `ct` / `merged` and 1e's `rates` / `CELL_ORDER` -- no recomputation,
# so the figure cannot drift from the tables above it.
if merged is None or merged["llm_met"].notna().sum() == 0:
    print("Sections 1d/1e did not produce a comparison; nothing to plot.")
else:
    # Two panels: what the labels do. Event incidence is section 1g's job --
    # it belongs to the analysis cohorts, not to this label comparison.
    fig, axes = plt.subplots(
        1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [1.15, 1]},
    )

    # --- (a) overlap: stacked bars, coverage gap explicit -------------------
    ax = axes[0]
    # Ordered ramp: the third step is an ABSENCE, not a peer category, so this
    # is sequential-with-a-gray-terminal rather than three categorical hues.
    OVERLAP_COLOR = {
        "LLM: metastatic": "#1f5fa8",
        "LLM: non-metastatic": "#7aa6d4",
        "(no LLM label)": "#c6c5c0",
    }
    seg_cols = [c for c in OVERLAP_COLOR if c in ct.columns]
    rows_a = [r for r in ("ADT: metastatic", "ADT: localized") if r in ct.index]
    left = np.zeros(len(rows_a))
    for col in seg_cols:
        vals = ct.loc[rows_a, col].to_numpy(dtype=float)
        ax.barh(rows_a, vals, left=left, height=0.6, color=OVERLAP_COLOR[col],
                label=col, edgecolor="white", linewidth=2)  # 2px surface gap
        row_total = ct.loc[rows_a, seg_cols].to_numpy().sum(axis=1).max()
        for y, (v, l) in enumerate(zip(vals, left)):
            if v <= 0:
                continue
            # A narrow segment is often the MOST interesting one (a small
            # disagreement group), so it gets its label outside the bar rather
            # than losing it to a width threshold.
            if v > 0.07 * row_total:
                ax.text(l + v / 2, y, f"{int(v):,}", ha="center", va="center",
                        fontsize=8,
                        color="white" if col != "(no LLM label)" else "#1a1a1a")
            else:
                ax.annotate(f"{int(v):,}", xy=(l + v / 2, y - 0.36),
                            ha="center", va="bottom", fontsize=7,
                            color="#52514e",
                            arrowprops=dict(arrowstyle="-", lw=0.6,
                                            color="#8f8e88", shrinkA=0, shrinkB=1))
        left += vals
    ax.set_xlabel("patients", fontsize=9)
    ax.set_title("(a) label overlap", fontsize=10, loc="left", fontweight="bold")
    ax.legend(fontsize=7.5, frameon=False, loc="lower right")
    ax.invert_yaxis()

    # --- (b) patients per agreement cell ------------------------------------
    ax = axes[1]
    cell_n = merged.loc[merged["llm_met"].notna(), "cell"] if "cell" in merged.columns else None
    if cell_n is None:
        both_f = merged.loc[merged["llm_met"].notna()].copy()
        both_f["cell"] = np.select(
            [both_f["adt_met"] & both_f["llm_met"].astype(bool),
             both_f["adt_met"] & ~both_f["llm_met"].astype(bool),
             ~both_f["adt_met"] & both_f["llm_met"].astype(bool)],
            ["both metastatic", "ADT only", "LLM only"],
            default="both non-metastatic")
        cell_n = both_f["cell"]
    # NOT `counts`: section 1b binds that name to the cohort x endpoint event
    # table and section 4a reads it back, so a generic name here silently
    # clobbers it. Notebook cells share one namespace -- keep new names scoped.
    cell_counts = cell_n.value_counts().reindex(CELL_ORDER).fillna(0).astype(int)
    # Agreement vs disagreement is the distinction that matters, so encode it:
    # one hue for the agreeing cells, one for the two disagreement cells.
    AGREE, DISAGREE = "#52514e", "#eb6834"
    bar_colors = [DISAGREE if c in ("ADT only", "LLM only") else AGREE
                  for c in cell_counts.index]
    ax.bar(range(len(cell_counts)), cell_counts.to_numpy(),
           color=bar_colors, width=0.68)
    for i, v in enumerate(cell_counts):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(cell_counts)))
    ax.set_xticklabels([c.replace(" ", "\n") for c in cell_counts.index],
                       fontsize=7.5)
    ax.set_ylabel("patients", fontsize=9)
    ax.set_title("(b) agreement cell size", fontsize=10, loc="left", fontweight="bold")
    ax.set_ylim(0, cell_counts.max() * 1.16 if cell_counts.max() else 1)
    # The agree/disagree split is encoded in the bar color; name it.
    ax.legend(handles=[
        Line2D([], [], marker="s", ls="none", ms=8, color=AGREE, label="labels agree"),
        Line2D([], [], marker="s", ls="none", ms=8, color=DISAGREE, label="labels disagree"),
    ], fontsize=7.5, frameon=False, loc="upper right")

    for ax in axes:
        ax.grid(axis="x" if ax is axes[0] else "y", alpha=0.25, lw=0.6)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)

    n_lab = int(merged["llm_met"].notna().sum())
    fig.suptitle(
        f"ADT-intent vs LLM metastatic labels | {len(merged):,} Stage 1 patients, "
        f"{n_lab:,} ({n_lab / len(merged):.0%}) jointly labelled",
        fontsize=11, y=1.04,
    )
    fig.tight_layout()
    save_supplement(fig, "figure_s_metastatic_label_overlap")
    plt.show()


### 1g. Event incidence across the analysis cohorts

Platinum and NEPC incidence in each of the 6 cohorts we actually model — the
cohort × endpoint grid from 1b, drawn. This is the figure that says how much
signal each cell of the cross has to work with.

Two panels, and the second is the one that matters for reading the first:

**(a) Event rate**, per cohort and endpoint, with 95% Wilson intervals. Wilson
rather than the normal approximation because the narrowed cohorts are small and
NEPC is rare — exactly the regime where a normal interval misbehaves and can
run below zero.

**(b) Cohort size and event count.** A rate of 12% means something different at
600 patients than at 40. Bars are patients; the darker overlay is events, so the
gap between them is the censored remainder.

Cohorts are in the section-4 narrowing cascade order, so reading down a column
follows the same progressive restriction that section 4 quantifies. Endpoints
keep the identity hues used by the section-3 forest plots.

A cell flagged in 1b as underpowered (< 25 events) is drawn with a hatched bar
here — the rate is still plotted, but it should be read as power, not biology.

In [ ]:
# --- 1g. Event incidence across the 6 analysis cohorts x 2 endpoints.
# Reads section 1b's `counts` -- the same table 1b displays, so the figure and
# that table cannot disagree.
def wilson(k, n, z=1.96):
    """Wilson score interval -- behaves at small n and near 0%, unlike normal."""
    if not n or not np.isfinite(k) or not np.isfinite(n):
        return (np.nan, np.nan)
    phat = k / n
    d = 1 + z**2 / n
    centre = (phat + z**2 / (2 * n)) / d
    half = z * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2)) / d
    return (100 * max(0.0, centre - half), 100 * min(1.0, centre + half))


THIN_EVENTS = 25  # same threshold section 1b flags on

if counts.empty or "status" not in counts.columns:
    print("Section 1b produced no counts; nothing to plot.")
else:
    # "ok" and "ok (<duration> absent)" are BOTH usable here: this figure needs
    # only n_patients / n_events, not the duration column. Matching .eq("ok")
    # would silently drop cohorts whose duration column is missing.
    inc = counts.loc[counts["status"].astype(str).str.startswith("ok")].copy()
    inc = inc.loc[inc["landmark_days"].eq(LANDMARK)]
    if inc.empty:
        print(f"No usable cohort x endpoint cells at landmark +{LANDMARK}d.")
    else:
        cohorts_g = [c for c in COHORT_ORDER if c in set(inc["cohort"].astype(str))]
        eps_g = [ep for ep in ENDPOINT_ORDER if ep in set(inc["endpoint"])]
        by = {(str(r["cohort"]), r["endpoint"]): r for _, r in inc.iterrows()}

        fig, axes = plt.subplots(1, 2, figsize=(13.5, 0.46 * len(cohorts_g) + 2.6),
                                 sharey=True, gridspec_kw={"width_ratios": [1, 1]})
        y = np.arange(len(cohorts_g))
        height = 0.8 / max(len(eps_g), 1)

        # --- (a) event rate with Wilson CIs ---
        ax = axes[0]
        for j, ep in enumerate(eps_g):
            color = ENDPOINT_COLOR.get(ep, FALLBACK_COLOR)
            vals, lo_err, hi_err, hatches = [], [], [], []
            for coh in cohorts_g:
                r = by.get((coh, ep))
                if r is None:
                    vals.append(np.nan); lo_err.append(0); hi_err.append(0)
                    hatches.append(False); continue
                n = float(r.get("n_patients", np.nan))
                k = float(r.get("n_events", np.nan))
                rate = float(r.get("event_rate_pct", np.nan))
                lo, hi = wilson(k, n)
                vals.append(rate)
                lo_err.append(max(0, rate - lo) if np.isfinite(lo) else 0)
                hi_err.append(max(0, hi - rate) if np.isfinite(hi) else 0)
                hatches.append(np.isfinite(k) and k < THIN_EVENTS)
            off = (j - (len(eps_g) - 1) / 2) * height
            bars = ax.barh(y + off, vals, height=height * 0.9, color=color, label=ep,
                           edgecolor="white", linewidth=1.2)
            # Hatch marks an underpowered cell; the bar still shows its rate.
            for b, thin in zip(bars, hatches):
                if thin:
                    b.set_hatch("///")
                    b.set_edgecolor("white")
            ax.errorbar(vals, y + off, xerr=[lo_err, hi_err], fmt="none",
                        ecolor="#1a1a1a", elinewidth=1, capsize=3, alpha=0.75)
            # Outside the bar, past the CI cap: an in-bar label is unreadable
            # over the hatch on thin cells, whatever color it is.
            for yi, v, he in zip(y + off, vals, hi_err):
                if np.isfinite(v):
                    ax.annotate(f"{v:.1f}", xy=(v + he, yi), xytext=(4, 0),
                                textcoords="offset points", ha="left",
                                va="center", fontsize=7, color="#52514e")
        ax.set_yticks(y)
        ax.set_yticklabels(cohorts_g, fontsize=8)
        ax.set_xlabel("event rate (%)", fontsize=9)
        ax.set_title(f"(a) incidence at landmark +{LANDMARK}d (95% Wilson CI)",
                     fontsize=10, loc="left", fontweight="bold")
        ax.set_xlim(0, ax.get_xlim()[1] * 1.14)  # room for the outside labels
        ax.legend(fontsize=8, frameon=False, loc="upper center",
                  bbox_to_anchor=(0.5, -0.13), ncol=len(eps_g))
        ax.invert_yaxis()

        # --- (b) denominator: patients, with events overlaid ---
        ax = axes[1]
        for j, ep in enumerate(eps_g):
            color = ENDPOINT_COLOR.get(ep, FALLBACK_COLOR)
            npat = [float(by.get((c, ep), {}).get("n_patients", np.nan))
                    for c in cohorts_g]
            nev = [float(by.get((c, ep), {}).get("n_events", np.nan))
                   for c in cohorts_g]
            off = (j - (len(eps_g) - 1) / 2) * height
            # Patients as a pale ground, events as the saturated overlay: the
            # visible remainder IS the censored/event-free group.
            ax.barh(y + off, npat, height=height * 0.9, color=color, alpha=0.28)
            ax.barh(y + off, nev, height=height * 0.9, color=color)
            for yi, (p_, e_) in zip(y + off, zip(npat, nev)):
                if np.isfinite(p_):
                    ax.annotate(f"{int(e_):,}/{int(p_):,}" if np.isfinite(e_)
                                else f"{int(p_):,}",
                                xy=(p_, yi), xytext=(3, 0), textcoords="offset points",
                                ha="left", va="center", fontsize=7, color="#52514e")
        ax.set_xlabel("patients (events overlaid)", fontsize=9)
        ax.set_title("(b) cohort size and event count", fontsize=10, loc="left",
                     fontweight="bold")
        # Endpoint identity is already established by panel (a); here the only
        # new distinction is pale-vs-solid, which one line explains.
        ax.annotate("pale = patients   solid = events   (label: events/patients)",
                    xy=(0.5, -0.13), xycoords="axes fraction", ha="center",
                    va="top", fontsize=7.5, color="#52514e")
        ax.set_xlim(0, ax.get_xlim()[1] * 1.20)

        for ax in axes:
            ax.grid(axis="x", alpha=0.25, lw=0.6)
            ax.set_axisbelow(True)
            for side in ("top", "right", "left"):
                ax.spines[side].set_visible(False)
            ax.tick_params(axis="y", length=0)

        thin_n = sum(
            1 for coh in cohorts_g for ep in eps_g
            if (r := by.get((coh, ep))) is not None
            and np.isfinite(float(r.get("n_events", np.nan)))
            and float(r["n_events"]) < THIN_EVENTS
        )
        sub = (f"hatched = < {THIN_EVENTS} events ({thin_n} of "
               f"{len(cohorts_g) * len(eps_g)} cells): read as power, not biology")
        fig.suptitle(
            f"Event incidence across the analysis cohorts | landmark +{LANDMARK}d\n"
            + sub, fontsize=11, y=1.06,
        )
        fig.tight_layout()
        save_supplement(fig, f"figure_s_cohort_event_incidence_landmark{LANDMARK}")
        plt.show()


## 2. PSA and testosterone: pre- vs post-treatment, stratified by event

Two lab sources, and the distinction matters:

- **Pre-treatment** values are lab rows drawn from strictly before the ADT
  anchor (`t_lab < 0`). The pre/post split stays at the anchor even though the
  cohort is the +180d one: at a +180d split, six months of on-treatment labs
  would land in the "pre-treatment" bucket and erase the contrast. The cohort
  membership — who is described — still comes from the +180d landmark file.
- **Post-treatment** values are therefore *not* in the aggregated files. They are
  read from the shared Stage 2 table `longitudinal_prediction_data_adt.csv`,
  whose `t_lab` is signed days from the ADT anchor and whose `LAB_VALUE` is
  already unit-standardized, then restricted to each cohort's MRNs.

`PSA` is the narrow OMOP-collapsed `LAB_NAME == "PSA"` series — the one that
drives the prediction features — not the broad PSA-code set used for cohort
prevalence. PSA is summarized on the median (and log1p mean) because it is
heavily right-skewed; testosterone on the median with a castrate fraction.

In [ ]:
import polars as pl

PSA_LAB = "PSA"
TESTOSTERONE_LAB = cp.TESTOSTERONE_LAB_NAME          # "Testosterone"
CASTRATE_NG_DL = cp.CASTRATE_TESTOSTERONE_NG_DL      # 50.0
LABS_OF_INTEREST = (PSA_LAB, TESTOSTERONE_LAB)

# Post-treatment window, in days from the ADT anchor. Bounded on the right so a
# "post" value is a treatment-response measurement rather than an arbitrarily
# late one; widen if you want the full follow-up.
POST_WINDOW = (1, 365)

LONGITUDINAL_CSV = DATA_ROOT / "longitudinal_prediction_data_adt.csv"


def load_longitudinal_labs():
    """Load PSA + testosterone from the shared Stage 2 longitudinal table.

    One load for all 6 cohorts: this table is anchor-level (built once per
    treatment anchor, before any cohort restriction), so every cohort is a row
    subset of it. Returns signed t_lab in days from the ADT anchor.
    """
    if not require(LONGITUDINAL_CSV, "Stage 2 longitudinal prediction data"):
        return None
    labs = (
        pl.scan_csv(LONGITUDINAL_CSV, infer_schema_length=0)
        .select("DFCI_MRN", "LAB_NAME", "LAB_VALUE", "t_lab")
        .with_columns(
            pl.col("DFCI_MRN").cast(pl.Float64, strict=False)
                             .cast(pl.Int64, strict=False),
            pl.col("LAB_VALUE").cast(pl.Float64, strict=False),
            pl.col("t_lab").cast(pl.Float64, strict=False),
        )
        .filter(
            pl.col("DFCI_MRN").is_not_null()
            & pl.col("LAB_NAME").is_in(list(LABS_OF_INTEREST))
            & pl.col("LAB_VALUE").is_not_null()
            & pl.col("t_lab").is_not_null()
        )
        .collect()
        .to_pandas()
    )
    labs[ID_COL] = normalize_id(labs[ID_COL])
    print(f"Longitudinal labs: {len(labs):,} rows, "
          f"{labs[ID_COL].nunique():,} patients, labs={sorted(labs['LAB_NAME'].unique())}")
    return labs


LABS = load_longitudinal_labs()

In [ ]:
def summarize_lab(values, lab_name):
    """Skew-aware summary for one lab series.

    PSA is heavily right-skewed, so the median is the headline and the mean is
    reported on the log1p scale; a raw PSA mean is dominated by a handful of
    very high values and is not comparable across cohorts.
    """
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if values.empty:
        return {}
    out = {
        "n_patients": int(len(values)),
        "median": float(values.median()),
        "q1": float(values.quantile(0.25)),
        "q3": float(values.quantile(0.75)),
        "mean": float(values.mean()),
    }
    if lab_name == PSA_LAB:
        out["mean_log1p"] = float(np.log1p(values.clip(lower=0)).mean())
    if lab_name == TESTOSTERONE_LAB:
        out["pct_castrate"] = float(100 * values.lt(CASTRATE_NG_DL).mean())
    return out


def per_patient_lab(labs, mrns, lab_name, window):
    """One value per patient: the median of that patient's results in `window`.

    Collapsing to a per-patient median first keeps a patient with 40 PSA draws
    from outweighing one with a single draw -- these are patient-level cohort
    descriptions, not observation-level ones.
    """
    lo, hi = window
    sub = labs.loc[
        labs["LAB_NAME"].eq(lab_name)
        & labs[ID_COL].isin(mrns)
        & labs["t_lab"].ge(lo)
        & labs["t_lab"].le(hi)
    ]
    if sub.empty:
        return pd.Series(dtype=float)
    return sub.groupby(ID_COL)["LAB_VALUE"].median()

In [ ]:
# --- 2a. Pre- vs post-treatment PSA/testosterone, stratified by event
#
# Two different landmarks are in play here, deliberately:
#   PRE_POST_LANDMARK selects WHICH COHORT is described -- the +180d cohort.
#   PRE_POST_BOUNDARY splits pre from post, and stays at the ANCHOR (0).
# They are separate because "pre-treatment" has to mean before ADT started. If
# the split followed the landmark, everything up to t_lab < 180 would count as
# "pre" and six months of on-treatment labs would be averaged into the
# pre-treatment number, erasing the contrast this section exists to show.
PRE_POST_LANDMARK = LANDMARK
PRE_POST_BOUNDARY = 0

prepost_rows = []
if LABS is not None:
    for run in RUNS:
        ep = run["endpoint"]
        event_col = EVENT_COL[ep]
        path = agg_path(run, PRE_POST_LANDMARK)
        if not path.exists():
            continue
        agg = pd.read_csv(path, low_memory=False)
        if event_col not in agg.columns or ID_COL not in agg.columns:
            continue
        agg[ID_COL] = normalize_id(agg[ID_COL])
        events = pd.to_numeric(agg[event_col], errors="coerce").fillna(0)

        pre_window = (-np.inf, PRE_POST_BOUNDARY - 1e-9)
        post_window = (max(POST_WINDOW[0], PRE_POST_BOUNDARY), POST_WINDOW[1])

        for lab in LABS_OF_INTEREST:
            for ev in (0, 1):
                mrns = set(agg.loc[events.eq(ev), ID_COL].dropna())
                if not mrns:
                    continue
                for period, window in (("pre", pre_window), ("post", post_window)):
                    stats = summarize_lab(per_patient_lab(LABS, mrns, lab, window), lab)
                    if not stats:
                        continue
                    prepost_rows.append({
                        "cohort": cohort_label(run), "endpoint": ep,
                        "lab": lab, "period": period,
                        f"{event_col}": ev, "event": ev,
                        "n_cohort": len(mrns), **stats,
                    })

prepost = pd.DataFrame(prepost_rows)
if not prepost.empty:
    prepost = order_cohorts(prepost)
    print(f"Cohort: landmark +{PRE_POST_LANDMARK}d   |   "
          f"Pre-window: t_lab < {PRE_POST_BOUNDARY}d   |   "
          f"Post-window: {max(POST_WINDOW[0], PRE_POST_BOUNDARY)}-{POST_WINDOW[1]}d "
          "from the ADT anchor\n")
    display(prepost.drop(columns=[c for c in prepost.columns if c in EVENT_COL.values()])
                   .round(2))
else:
    print("No pre/post lab summaries could be built; see missing artifacts.")

In [ ]:
# Headline view: median by event status, pre vs post, one block per lab.
if not prepost.empty:
    for lab in LABS_OF_INTEREST:
        sub = prepost.loc[prepost["lab"].eq(lab)]
        if sub.empty:
            continue
        grid = sub.pivot_table(
            index=["cohort", "endpoint"], columns=["period", "event"],
            values="median", observed=False,
        )
        # pre before post, event 0 before 1
        grid = grid.reindex(columns=pd.MultiIndex.from_product(
            [["pre", "post"], [0, 1]], names=["period", "event"]
        ), fill_value=np.nan).dropna(how="all")
        print(f"\n=== {lab}: median per-patient value, by event status ===")
        display(grid.round(2))

    # Testosterone castrate fraction is the direct read on ADT effectiveness.
    t_sub = prepost.loc[prepost["lab"].eq(TESTOSTERONE_LAB)]
    if not t_sub.empty and "pct_castrate" in t_sub.columns:
        grid = t_sub.pivot_table(
            index=["cohort", "endpoint"], columns=["period", "event"],
            values="pct_castrate", observed=False,
        )
        # pivot_table sorts columns alphabetically, which puts "post" before
        # "pre"; reindex so the table reads in time order.
        grid = grid.reindex(columns=pd.MultiIndex.from_product(
            [["pre", "post"], [0, 1]], names=["period", "event"]
        ), fill_value=np.nan).dropna(how="all")
        print(f"\n=== Testosterone: % of patients below {CASTRATE_NG_DL:g} ng/dL ===")
        display(grid.round(1))

### 2b. Within-patient pre → post change

The tables above compare two independently-summarized groups. This one pairs
each patient with themselves — patients contributing both a pre- and a
post-treatment value — which is the comparison that actually speaks to
treatment response, and tests whether the change differs by event status.

In [ ]:
from scipy import stats as sps

delta_rows = []
if LABS is not None:
    for run in RUNS:
        ep = run["endpoint"]
        event_col = EVENT_COL[ep]
        path = agg_path(run, PRE_POST_LANDMARK)
        if not path.exists():
            continue
        agg = pd.read_csv(path, low_memory=False)
        if event_col not in agg.columns or ID_COL not in agg.columns:
            continue
        agg[ID_COL] = normalize_id(agg[ID_COL])
        events = pd.to_numeric(agg[event_col], errors="coerce").fillna(0)
        event_by_mrn = pd.Series(events.values, index=agg[ID_COL].values)
        all_mrns = set(agg[ID_COL].dropna())

        for lab in LABS_OF_INTEREST:
            pre = per_patient_lab(LABS, all_mrns, lab, (-np.inf, PRE_POST_BOUNDARY - 1e-9))
            post = per_patient_lab(
                LABS, all_mrns, lab,
                (max(POST_WINDOW[0], PRE_POST_BOUNDARY), POST_WINDOW[1]),
            )
            paired = pd.DataFrame({"pre": pre, "post": post}).dropna()
            if paired.empty:
                continue
            paired["event"] = paired.index.map(event_by_mrn)
            paired = paired.dropna(subset=["event"])
            # PSA change on the log1p scale: a drop from 100 -> 10 and 10 -> 1
            # are the same biological response, and only the log sees that.
            if lab == PSA_LAB:
                paired["delta"] = (np.log1p(paired["post"].clip(lower=0))
                                   - np.log1p(paired["pre"].clip(lower=0)))
                delta_units = "log1p(ng/mL)"
            else:
                paired["delta"] = paired["post"] - paired["pre"]
                delta_units = "ng/dL"

            groups = {ev: g["delta"].dropna() for ev, g in paired.groupby("event")}
            row = {
                "cohort": cohort_label(run), "endpoint": ep, "lab": lab,
                "delta_units": delta_units, "n_paired": int(len(paired)),
            }
            for ev in (0, 1):
                g = groups.get(ev, pd.Series(dtype=float))
                row[f"n_event{ev}"] = int(len(g))
                row[f"median_delta_event{ev}"] = float(g.median()) if len(g) else np.nan
            g0, g1 = groups.get(0, pd.Series(dtype=float)), groups.get(1, pd.Series(dtype=float))
            # Mann-Whitney: does the pre->post change differ by event status?
            # Descriptive only -- unadjusted, and the cells are not independent.
            if len(g0) >= 3 and len(g1) >= 3:
                row["p_delta_differs"] = float(
                    sps.mannwhitneyu(g0, g1, alternative="two-sided").pvalue
                )
            else:
                row["p_delta_differs"] = np.nan
            delta_rows.append(row)

deltas = pd.DataFrame(delta_rows)
if not deltas.empty:
    deltas = order_cohorts(deltas)
    print("Within-patient pre -> post change (patients with both windows observed).\n"
          "p-value: Mann-Whitney on the change, event=1 vs event=0. Unadjusted.\n")
    display(deltas.round(3))
else:
    print("No patients had both a pre- and a post-treatment value.")

## 3. Univariate associations across endpoints and cohorts

Hazard ratios per SD from the fitted univariate Cox models
(`cox_agg_univariate_nobs_adjusted.csv`), loaded with the same
`compass_pipeline.load_univariate_results` the other notebooks use.

Section 3a is PSA and testosterone specifically; 3b is the full feature set, for
context on whether the lab signal is specific or part of a broad association.

**Read these against section 1.** A feature nominally significant for platinum
but not NEPC is not evidence of a real difference when the NEPC arm has a
fraction of the events — the confidence intervals carry that information, so
they are kept in every table.

In [ ]:
HR_COL = "hazard_ratio_per_sd"

univariate_frames = []
for run in RUNS:
    try:
        frame = cp.load_univariate_results(run)
    except (FileNotFoundError, ValueError) as exc:
        MISSING.append(f"{run['label']} [{run['endpoint']}] univariate: {exc}")
        continue
    # load_univariate_results stamps `cohort` with the run LABEL (adt_..._noprecastrate);
    # relabel to this notebook's cohort/endpoint axes so tables pivot cleanly.
    frame["run_label"] = run["label"]
    frame["cohort"] = cohort_label(run)
    frame["endpoint"] = run["endpoint"]
    univariate_frames.append(frame)

if univariate_frames:
    univariate = pd.concat(univariate_frames, ignore_index=True)
    for col in (HR_COL, "ci_lower", "ci_upper", "p_value", "q_value"):
        if col in univariate.columns:
            univariate[col] = pd.to_numeric(univariate[col], errors="coerce")
    # n_observations counts how often a lab was DRAWN, not what it measured: it
    # is a surveillance-intensity proxy (sicker patients get tested more), not a
    # property of the analyte. Dropped once here so every view below -- the 3a/3b
    # tables, the 3c forest plot and the section-4 cascade -- excludes it
    # consistently rather than each filtering on its own.
    DROP_STATS = {"n_observations"}
    stat = (univariate["feature_stat"] if "feature_stat" in univariate.columns
            else univariate["feature"].astype(str).str.split("__").str[-1])
    n_dropped = int(stat.astype(str).isin(DROP_STATS).sum())
    univariate = univariate.loc[~stat.astype(str).isin(DROP_STATS)].copy()
    print(f"Dropped {n_dropped:,} {sorted(DROP_STATS)} row(s) as a "
          "surveillance-intensity proxy, not a lab effect.")
    print(f"Univariate rows: {len(univariate):,} across "
          f"{univariate['cohort'].nunique()} cohorts x {univariate['endpoint'].nunique()} endpoints")
    print("landmarks:", sorted(univariate["landmark_days"].dropna().unique()))
else:
    univariate = pd.DataFrame()
    print("No univariate results found; run 02_univariate for this cross first.")

In [ ]:
# --- 3a. PSA and testosterone only
def is_lab_of_interest(frame):
    """Match PSA/testosterone rows by lab_name when present, else by feature name.

    Matches PSA_raw / PSA_log1p as well as the plain PSA series, so the scale
    supplement's features are not silently dropped from this view.
    """
    if "lab_name" in frame.columns:
        name = frame["lab_name"].astype(str)
    else:
        name = frame["feature"].astype(str).str.split("__").str[0]
    lowered = name.str.lower()
    return lowered.str.startswith("psa") | lowered.str.contains("testosterone")


lab_univariate = pd.DataFrame()
if not univariate.empty:
    lab_univariate = univariate.loc[is_lab_of_interest(univariate)].copy()
    keep = [c for c in [
        "cohort", "endpoint", "landmark_days", "feature", "lab_name", "feature_stat",
        "n_patients_used", "n_events_used", HR_COL, "ci_lower", "ci_upper",
        "p_value", "q_value",
    ] if c in lab_univariate.columns]
    lab_univariate = order_cohorts(lab_univariate[keep])
    if lab_univariate.empty:
        print("No PSA/testosterone rows in the univariate results.")
    else:
        display(lab_univariate.sort_values(
            ["cohort", "endpoint", "landmark_days", "p_value"]
        ).round(4))

In [ ]:
# HR grid: one row per lab feature, columns = cohort x endpoint, at one landmark.
UNIVARIATE_LANDMARK = LANDMARK

if not lab_univariate.empty:
    at_lm = lab_univariate.loc[lab_univariate["landmark_days"].eq(UNIVARIATE_LANDMARK)]
    if not at_lm.empty:
        for value, title in ((HR_COL, "hazard ratio per SD"), ("p_value", "p-value")):
            grid = at_lm.pivot_table(
                index="feature", columns=["endpoint", "cohort"],
                values=value, observed=False,
            )
            print(f"\n=== {title} | landmark +{UNIVARIATE_LANDMARK}d ===")
            display(grid.round(4))

        # Nominal significance as a count, so a feature that holds up across the
        # whole cross is distinguishable from one significant in a single cell.
        sig = (
            at_lm.assign(nominal=at_lm["p_value"].lt(0.05))
            .groupby(["feature", "endpoint"], observed=False)["nominal"]
            .agg(n_cohorts_nominal="sum", n_cohorts="size")
            .reset_index()
        )
        print(f"\n=== nominal significance (p < 0.05) across the {len(COHORT_ORDER)} "
              f"cohorts | landmark +{UNIVARIATE_LANDMARK}d ===")
        display(sig.sort_values(["endpoint", "n_cohorts_nominal"], ascending=[True, False]))
    else:
        print(f"No PSA/testosterone rows at landmark +{UNIVARIATE_LANDMARK}d.")

### 3c. Forest plots: PSA and testosterone across cohorts

One figure per analyte — PSA and testosterone are on different clinical
footings and get read separately, so they no longer share a canvas. Within each
figure, rows are cohorts and columns are endpoints; where an analyte has several
feature variants (`PSA_raw`, `PSA_log1p`, …) each becomes its own row band.

Hazard ratio per SD with 95% confidence intervals. The HR axis is
**log-scaled** — the natural scale for a ratio, where HR 0.5 and HR 2.0 sit
equal distances from the null. The x-range is shared **across both figures**, so
a PSA effect and a testosterone effect of the same size are the same distance
from the null line in either one.

`n_observations` features are excluded from **all** of sections 3-4 (dropped
where the univariate results are loaded, so tables and figures agree). It counts
how many times a lab was drawn — a measure of clinical attention, since sicker
patients get tested more — not a property of the analyte. It behaves as a
surveillance-intensity confounder and does not belong beside real
PSA/testosterone effects.

Marker fill encodes nominal significance (solid `p < 0.05`, hollow otherwise);
a point whose interval crosses the dashed null line at HR = 1 is not
distinguishable from no effect. Read alongside section 1 — a wide interval in a
narrowed cohort is thin events, not a real null.

Both figures are written to `SUPPLEMENT_DIR` (see the cell below) at 600 dpi,
matching the DPI and the `<arm>/` layout the R figure pipeline uses.

In [ ]:
def _hr_fmt(x, _pos):
    """Hazard ratios as plain numbers: 0.5, 1, 2 -- never 6x10^-1."""
    if x <= 0:
        return ""
    return f"{x:g}" if x >= 1 else f"{x:.2f}".rstrip("0").rstrip(".")



def _lab_names(frame):
    """The analyte behind each row -- same rule as is_lab_of_interest()."""
    if "lab_name" in frame.columns:
        return frame["lab_name"].astype(str)
    return frame["feature"].astype(str).str.split("__").str[0]


def _forest_frame():
    """Rows to plot: PSA/testosterone univariate results at the landmark.

    n_observations is already excluded upstream, where the univariate results
    are loaded, so every table and figure in sections 3-4 shares one filter.
    """
    if lab_univariate.empty:
        return pd.DataFrame()
    frame = lab_univariate.loc[
        lab_univariate["landmark_days"].eq(UNIVARIATE_LANDMARK)
    ].copy()
    frame = frame.loc[frame[HR_COL].notna()]
    if frame.empty:
        return frame
    lowered = _lab_names(frame).str.lower()
    # One analyte per figure. Anything matching neither is reported, not dropped
    # silently -- is_lab_of_interest() admits any "psa*"/"*testosterone*" name.
    frame["analyte"] = np.select(
        [lowered.str.startswith("psa"), lowered.str.contains("testosterone")],
        ["PSA", "Testosterone"],
        default="other",
    )
    return frame


forest = _forest_frame()
if forest.empty:
    print(f"No PSA/testosterone rows to plot at landmark +{UNIVARIATE_LANDMARK}d.")
else:
    stray = forest.loc[forest["analyte"].eq("other")]
    if not stray.empty:
        print(f"!! {len(stray)} row(s) matched neither analyte and are not plotted: "
              f"{sorted(set(_lab_names(stray)))}")
        forest = forest.loc[~forest["analyte"].eq("other")]

    endpoints = [ep for ep in ENDPOINT_ORDER if ep in set(forest["endpoint"])]
    # Cohorts on the y-axis, in the section-4 cascade order, top to bottom.
    cohorts = [c for c in COHORT_ORDER if c in set(forest["cohort"].astype(str))]
    ypos = {c: len(cohorts) - 1 - i for i, c in enumerate(cohorts)}

    # One shared x-range across BOTH figures, so an HR in the PSA figure and the
    # same HR in the testosterone figure sit at the same place on the page.
    finite = pd.concat([
        pd.to_numeric(forest[c], errors="coerce")
        for c in (HR_COL, "ci_lower", "ci_upper") if c in forest.columns
    ]).replace([np.inf, -np.inf], np.nan).dropna()
    finite = finite.loc[finite.gt(0)]
    if finite.empty:
        xlim = (0.5, 2.0)
    else:
        xlim = (float(finite.min()) / 1.15, float(finite.max()) * 1.15)

    def forest_figure(analyte):
        """One forest figure for a single analyte; returns (fig, stem) or None."""
        sub = forest.loc[forest["analyte"].eq(analyte)]
        if sub.empty:
            print(f"No {analyte} rows at landmark +{UNIVARIATE_LANDMARK}d; skipped.")
            return None
        features = sorted(sub["feature"].astype(str).unique())
        eps = [ep for ep in endpoints if ep in set(sub["endpoint"])]
        if not eps:
            print(f"No endpoints with {analyte} rows; skipped.")
            return None

        fig, axes = plt.subplots(
            len(features), len(eps),
            figsize=(4.6 * len(eps), 0.42 * len(cohorts) * len(features) + 1.6),
            squeeze=False, sharex=True, sharey=True,
        )
        for r, feature in enumerate(features):
            for c, ep in enumerate(eps):
                ax = axes[r][c]
                color = ENDPOINT_COLOR.get(ep, FALLBACK_COLOR)
                block = sub.loc[
                    sub["feature"].astype(str).eq(feature) & sub["endpoint"].eq(ep)
                ]
                # Null line first, so data marks sit on top of it.
                ax.axvline(1.0, color="#8f8e88", lw=1, ls="--", zorder=1)
                for _, row in block.iterrows():
                    y = ypos.get(str(row["cohort"]))
                    if y is None:
                        continue
                    lo, hi = row.get("ci_lower"), row.get("ci_upper")
                    if pd.notna(lo) and pd.notna(hi):
                        ax.plot([lo, hi], [y, y], color=color, lw=2,
                                solid_capstyle="round", zorder=2)
                    sig = pd.notna(row.get("p_value")) and row["p_value"] < 0.05
                    # Fill encodes significance so it is not carried by color alone.
                    ax.plot(row[HR_COL], y, "o", ms=8, zorder=3,
                            color=color if sig else "none",
                            markerfacecolor=color if sig else "none",
                            markeredgecolor=color, markeredgewidth=2)
                ax.set_yticks(range(len(cohorts)))
                ax.set_yticklabels(list(reversed(cohorts)), fontsize=8)
                ax.set_ylim(-0.6, len(cohorts) - 0.4)
                ax.set_xscale("log")
                # A log axis defaults to one labelled tick per decade, which on
                # an HR range near 1 means the ONLY number on the axis is "1".
                # Label the 1-2-5 subdivisions too, so intervals can be read.
                ax.xaxis.set_major_locator(
                    mticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0), numticks=12)
                )
                ax.xaxis.set_minor_locator(
                    mticker.LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1,
                                       numticks=12)
                )
                # Plain HR values, not 6x10^-1 scientific notation: these are
                # ratios and readers compare them against 1 directly.
                ax.xaxis.set_major_formatter(mticker.FuncFormatter(_hr_fmt))
                ax.xaxis.set_minor_formatter(mticker.NullFormatter())
                ax.tick_params(axis="y", length=0)  # labels carry the rows
                ax.grid(axis="x", alpha=0.25, lw=0.6)
                ax.set_axisbelow(True)
                for side in ("top", "right", "left"):
                    ax.spines[side].set_visible(False)
                if r == 0:
                    ax.set_title(ep, fontsize=10, color=color, pad=8)
                if r == len(features) - 1:
                    ax.set_xlabel("hazard ratio per SD (log scale)", fontsize=9)
                if c == 0 and len(features) > 1:
                    # Sits above the panel title on row 0, so the two never
                    # share a line.
                    ax.annotate(feature, xy=(0, 1.10 if r == 0 else 1.02),
                                xycoords="axes fraction",
                                fontsize=9, fontweight="bold", va="bottom")
                ax.set_xlim(*xlim)

        handles = [
            Line2D([], [], marker="o", ls="none", ms=8,
                   color=ENDPOINT_COLOR.get(ep, FALLBACK_COLOR), label=ep)
            for ep in eps
        ] + [
            Line2D([], [], marker="o", ls="none", ms=8, color="#52514e",
                   label="p < 0.05"),
            Line2D([], [], marker="o", ls="none", ms=8, markerfacecolor="none",
                   markeredgecolor="#52514e", markeredgewidth=2, color="none",
                   label="not nominally significant"),
        ]
        fig.legend(handles=handles, loc="lower center", ncol=len(handles),
                   frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.02))
        fig.suptitle(
            f"{analyte} univariate associations | landmark +{UNIVARIATE_LANDMARK}d",
            fontsize=11, y=1.0,
        )
        fig.tight_layout()
        stem = (f"figure_s_forest_{analyte.lower()}"
                f"_landmark{UNIVARIATE_LANDMARK}")
        return fig, stem

    for analyte in ("PSA", "Testosterone"):
        made = forest_figure(analyte)
        if made is None:
            continue
        fig, stem = made
        save_supplement(fig, stem)
        plt.show()

    # Table view: the same numbers the figures encode, for exact values.
    show = [c for c in ["analyte", "cohort", "endpoint", "feature", "n_events_used",
                        HR_COL, "ci_lower", "ci_upper", "p_value", "q_value"]
            if c in forest.columns]
    display(forest[show].sort_values(["analyte", "endpoint", "feature", "cohort"]).round(4))


In [ ]:
# --- 3b. Full feature set: strongest associations per cohort x endpoint
if not univariate.empty:
    at_lm = univariate.loc[univariate["landmark_days"].eq(UNIVARIATE_LANDMARK)].copy()
    if not at_lm.empty:
        TOP_N = 8
        top = (
            at_lm.dropna(subset=["p_value"])
            .sort_values("p_value")
            .groupby(["cohort", "endpoint"], observed=False)
            .head(TOP_N)
        )
        cols = [c for c in ["cohort", "endpoint", "feature", "n_events_used",
                            HR_COL, "ci_lower", "ci_upper", "p_value", "q_value"]
                if c in top.columns]
        print(f"Top {TOP_N} features by p-value per cell | landmark +{UNIVARIATE_LANDMARK}d")
        display(order_cohorts(top[cols]).round(4))

        # How many features clear nominal / FDR thresholds in each cell -- the
        # denominator for reading any single association above.
        summary = at_lm.groupby(["cohort", "endpoint"], observed=False).apply(
            lambda g: pd.Series({
                "n_features": len(g),
                "n_nominal_p05": int(g["p_value"].lt(0.05).sum()),
                "n_fdr_q05": int(g["q_value"].lt(0.05).sum())
                             if "q_value" in g.columns else np.nan,
                "min_p": g["p_value"].min(),
                "median_n_events_used": g["n_events_used"].median()
                                        if "n_events_used" in g.columns else np.nan,
            }),
            include_groups=False,
        ).reset_index()
        print("\n=== association yield per cell ===")
        display(order_cohorts(summary).round(4))

## 4. Effect of narrowing the cohort

The cohort axis is **nested**: `metastatic_*` is a subset of `all`, and
`+noprecastrate` removes pre-ADT-castrate patients from whichever cohort it
composes onto. This section walks that cascade so a change in an estimate is
never read as biology when it is really a change in who is being counted.

Two independent metastatic definitions are compared at the same step of the
cascade — `metastatic_adt` (medication-derived ADT intent) and
`metastatic_llm` (met_diagnosis LLM adjudication) — so a result that only
appears under one of them is visible as such.

In [ ]:
# --- 4a. Attrition down the cascade, per endpoint x landmark
BASELINE_COHORT = "all"

cascade_rows = []
if not counts.empty and "status" in counts.columns:
    ok = counts.loc[counts["status"].eq("ok")].copy()
    ok["cohort"] = ok["cohort"].astype(str)
    for (ep, lm), block in ok.groupby(["endpoint", "landmark_days"], observed=False):
        block = block.set_index("cohort")
        if BASELINE_COHORT not in block.index:
            continue
        base = block.loc[BASELINE_COHORT]
        for name, row in block.iterrows():
            cascade_rows.append({
                "cohort": name, "endpoint": ep, "landmark_days": lm,
                "n_patients": row["n_patients"], "n_events": row["n_events"],
                "event_rate_pct": row["event_rate_pct"],
                "pct_of_all_patients": 100 * row["n_patients"] / base["n_patients"]
                                       if base["n_patients"] else np.nan,
                "pct_of_all_events": 100 * row["n_events"] / base["n_events"]
                                      if base["n_events"] else np.nan,
                "event_rate_ratio_vs_all": (row["event_rate_pct"] / base["event_rate_pct"]
                                            if base["event_rate_pct"] else np.nan),
            })

cascade = pd.DataFrame(cascade_rows)
if not cascade.empty:
    cascade = order_cohorts(cascade)
    print(f"Retention and event enrichment relative to cohort '{BASELINE_COHORT}'.\n"
          "event_rate_ratio_vs_all > 1 means the narrower cohort is event-enriched.\n")
    display(cascade.round(2))
else:
    print("Cohort counts are needed for the cascade table; see section 1.")

In [ ]:
# --- 4b. The two narrowing steps, isolated
#
# Step 1 (all -> metastatic) and step 2 (+noprecastrate) are reported
# separately: the exclusion composes onto EVERY cohort, so its effect is only
# interpretable within a fixed cohort, not against `all`.
step_rows = []
if not cascade.empty:
    idx = cascade.set_index(["cohort", "endpoint", "landmark_days"])
    for ep in ENDPOINTS:
        for lm in LANDMARKS:
            def get(c):
                try:
                    return idx.loc[(c, ep, lm)]
                except KeyError:
                    return None

            # Step 1: metastatic restriction, within exclusion="none".
            base = get("all")
            for met in ("metastatic_adt", "metastatic_llm"):
                row = get(met)
                if base is None or row is None:
                    continue
                step_rows.append({
                    "step": f"all -> {met}", "endpoint": ep, "landmark_days": lm,
                    "n_before": base["n_patients"], "n_after": row["n_patients"],
                    "pct_patients_retained": 100 * row["n_patients"] / base["n_patients"]
                                              if base["n_patients"] else np.nan,
                    "events_before": base["n_events"], "events_after": row["n_events"],
                    "rate_before_pct": base["event_rate_pct"],
                    "rate_after_pct": row["event_rate_pct"],
                    "rate_ratio": (row["event_rate_pct"] / base["event_rate_pct"]
                                   if base["event_rate_pct"] else np.nan),
                })
            # Step 2: the pre-ADT-castrate exclusion, within each cohort.
            for coh in COHORTS:
                before, after = get(coh), get(f"{coh} +noprecastrate")
                if before is None or after is None:
                    continue
                step_rows.append({
                    "step": f"{coh} -> +noprecastrate", "endpoint": ep, "landmark_days": lm,
                    "n_before": before["n_patients"], "n_after": after["n_patients"],
                    "pct_patients_retained": 100 * after["n_patients"] / before["n_patients"]
                                              if before["n_patients"] else np.nan,
                    "events_before": before["n_events"], "events_after": after["n_events"],
                    "rate_before_pct": before["event_rate_pct"],
                    "rate_after_pct": after["event_rate_pct"],
                    "rate_ratio": (after["event_rate_pct"] / before["event_rate_pct"]
                                   if before["event_rate_pct"] else np.nan),
                })

steps = pd.DataFrame(step_rows)
if not steps.empty:
    display(steps.round(2))

In [ ]:
# --- 4c. Does the PSA/testosterone association survive the narrowing?
#
# The same feature tracked down the cascade. A hazard ratio that moves while its
# confidence interval widens is attrition, not a cohort effect; one that moves
# while the interval holds is worth a second look.
if not lab_univariate.empty:
    at_lm = lab_univariate.loc[lab_univariate["landmark_days"].eq(UNIVARIATE_LANDMARK)]
    if not at_lm.empty:
        for ep in ENDPOINTS:
            block = at_lm.loc[at_lm["endpoint"].eq(ep)]
            if block.empty:
                continue
            wide = block.pivot_table(
                index="feature", columns="cohort",
                values=[HR_COL, "p_value", "n_events_used"], observed=False,
            )
            wide = wide.reindex(
                columns=pd.MultiIndex.from_product(
                    [[HR_COL, "p_value", "n_events_used"], COHORT_ORDER]
                ), fill_value=np.nan,
            ).dropna(how="all").dropna(axis=1, how="all")
            print(f"\n=== {ep}: PSA/testosterone across the cascade "
                  f"| landmark +{UNIVARIATE_LANDMARK}d ===")
            display(wide.round(4))

In [ ]:
# --- 4d. Kaplan-Meier down the cascade, one panel per endpoint
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

KM_LANDMARK = LANDMARK

fig, axes = plt.subplots(1, len(ENDPOINTS), figsize=(7 * len(ENDPOINTS), 5), squeeze=False)
drew_any = False
for ax, ep in zip(axes[0], ENDPOINTS):
    event_col, duration_col = EVENT_COL[ep], DURATION_COL[ep]
    for run in RUNS:
        if run["endpoint"] != ep:
            continue
        path = agg_path(run, KM_LANDMARK)
        if not path.exists():
            continue
        agg = pd.read_csv(path, low_memory=False)
        if event_col not in agg.columns or duration_col not in agg.columns:
            continue
        durations = pd.to_numeric(agg[duration_col], errors="coerce")
        events = pd.to_numeric(agg[event_col], errors="coerce").fillna(0)
        valid = durations.notna() & durations.gt(0)
        if not valid.any():
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(
            durations.loc[valid], events.loc[valid],
            label=f"{cohort_label(run)} (n={int(valid.sum())}, "
                  f"e={int(events.loc[valid].eq(1).sum())})",
        )
        kmf.plot_survival_function(ax=ax, ci_show=False)
        drew_any = True
    ax.set_title(f"{ep}-free survival | landmark +{KM_LANDMARK}d")
    ax.set_xlabel("Days from ADT start")
    ax.set_ylabel("Event-free probability")
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)

if drew_any:
    plt.tight_layout()
    plt.show()
else:
    plt.close(fig)
    print("No cohort had a usable landmark file for the KM panels.")

## Missing artifacts

In [ ]:
if MISSING:
    print(f"{len(MISSING)} artifact(s) were not found:")
    for item in dict.fromkeys(MISSING):
        print(f"  {item}")
    print(
        "\nRun 01/02/03 with the same ARMS/ENDPOINTS/COHORTS/EXCLUSIONS as the "
        "config cell above; each builds one tree per cohort x endpoint x exclusion."
    )
else:
    print("All expected artifacts were found.")